## PRebuilt
- stdio (npx/uvx)
- http

In [3]:
# 1.加载环境变量
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
qwen_model = init_chat_model("openai:qwen3.7-plus", temperature=0, base_url=os.getenv("ROOT_URL"))

# 3.定义工具，用MCP获取工具
# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

# 3.2.用client拉取tool
# 注意MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools = await client.get_tools()


# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=qwen_model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("现在是什么时间")]}
)
print(response)

{'messages': [HumanMessage(content='现在是什么时间', additional_kwargs={}, response_metadata={}, id='222a9e12-f7d1-4750-bef8-47636e477674'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 503, 'total_tokens': 580, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 46, 'rejected_prediction_tokens': None, 'text_tokens': 77}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 503}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-e9877246-0e40-9ac7-bbe9-184cd92cf5fe', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1010c-7bc4-79e1-b280-b82f94de9623-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Shanghai'}, 'id': 'call_d44917e1112d40259539eff4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接Kiwi MCP服务器
client = MultiServerMCPClient(
    {
        "kiwi-com-flight-search": {
            "transport": "http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

# 注意MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools = await client.get_tools()


agent = create_agent(
    model=qwen_model,
    tools=tools,
    system_prompt="You are a travel agent. Help user find best flights. No follow up questions. If the user uses Chinese, use zh-cn as the locale value."
)

response = await agent.ainvoke({
    "messages": [HumanMessage("查一下2026年10月16日晚上从香港飞上海虹桥的航班。")]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

查一下2026年10月16日晚上从香港飞上海虹桥的航班。
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_3dbf75d04c2c492cb61e335d)
 Call ID: call_3dbf75d04c2c492cb61e335d
  Args:
    departureDate: 16/10/2026
    dtime_from: 18
    flyFrom: 香港
    flyTo: 上海虹桥
    locale: zh-cn
================================= Tool Message =================================
Name: search-flight

[{'type': 'text', 'text': '{\n  "query": "香港 → 上海虹桥 on 16/10/2026, 1 adult [dep from 18:00]",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\n    "children": 0,\n    "infants": 0\n  },\n  "resultsCount": 0,\n  "itineraries": [],\n  "searchTimeMs": 203,\n  "error": null\n}', 'id': 'lc_96e8f629-10a3-46bd-b03d-0a3f03234544'}]
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_3211a7e0791245d98717eb22)
 Call ID: call_3211a7

## Customized

In [ ]:
# from fastmcp import FastMCP

# # 初始化mcp
# mcp = FastMCP("Math")


# # mcp tools
# @mcp.tool()
# def add(a: float, b: float) -> float:
#     """Add two numbers"""
#     return a + b


# @mcp.tool()
# def multiply(a: float, b: float) -> float:
#     """Multiply two numbers"""
#     return a * b


# @mcp.tool
# def square_root(x: float) -> float:
#     """Calculate the square root of a number"""
#     return x ** 0.5


# # 启动mcp服务，通信方式设置为stdio
# if __name__ == "__main__":
#     mcp.run(transport="stdio")

In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 连接自定义MCP服务
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["./my_mcps/math_mcp_server.py"] # 这里是脚本所在的路径，此处是相对路径
        }
    }
)
# 获取工具
tools = await client.get_tools()

# 创建agent
agent = create_agent(
    model=qwen_model,
    tools=tools,
    system_prompt="You are a helpful agent. You must use tools to answer math question."
)

# 调用测试
response = await agent.ainvoke({
    "messages": [HumanMessage("467和529的平方根之和是多少?")]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

467和529的平方根之和是多少?
================================== Ai Message ==================================
Tool Calls:
  square_root (call_91c9778b56e64c4bbc4677c0)
 Call ID: call_91c9778b56e64c4bbc4677c0
  Args:
    x: 467
  square_root (call_1e3d752e189b4a51b9fe5d25)
 Call ID: call_1e3d752e189b4a51b9fe5d25
  Args:
    x: 529
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '21.61018278497431', 'id': 'lc_7a2c1c29-0ef7-4499-a171-f07f01668101'}]
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '23.0', 'id': 'lc_2ab19d23-5946-4c37-bcf4-9bad9d437553'}]
================================== Ai Message ==================================
Tool Calls:
  add (call_4cec6ca322214f7dba9cb886)
 Call ID: call_4cec6ca322214f7dba9cb886
  Args:
    a: 21.61018278497431
    b: 23
==